# Create new Salesforce Job__c (TEST)

POST a new `Job__c` record from Supabase `job_current`. Records are automatically prefixed with `[TEST]` and marked with `Job_Status__c = "Test"` so they are clearly identifiable as test data.

- Set **`SUPABASE_JOB_ID`** below, then run all cells.
- `DRY_RUN = True` previews the payload without writing.

In [2]:
import os, sys, json
from pathlib import Path

project_root = Path.cwd().resolve()
for _ in range(15):
    if (project_root / "src" / "utils").is_dir():
        break
    project_root = project_root.parent
else:
    raise RuntimeError("Could not locate repo root containing src/utils")

sys.path.insert(0, str(project_root / "src"))
from dotenv import load_dotenv
load_dotenv(project_root / ".env")
print("Ready")

Ready


In [7]:
# --- Change these ---
SUPABASE_JOB_ID = "19448"
SUPABASE_SCHEMA = "public"
DRY_RUN = False

In [8]:
from utils.supabase_db import load_job_current_row_for_salesforce

job_row = load_job_current_row_for_salesforce(SUPABASE_JOB_ID, schema=SUPABASE_SCHEMA)
print(f"Loaded job_id={job_row.get('job_id')}  city={job_row.get('city')}  state={job_row.get('state')}")

Loaded job_id=19448  city=Syracuse  state=NY


In [9]:
from utils.salesforce import get_token_auto

token = get_token_auto(
    os.environ["SALESFORCE_CONSUMER_KEY"],
    os.environ["SALESFORCE_CONSUMER_SECRET"],
    os.environ.get("SALESFORCE_USERNAME") or None,
    os.environ.get("SALESFORCE_PASSWORD") or None,
    use_client_credentials=os.environ.get("SALESFORCE_USE_USERNAME_PASSWORD", "").lower() not in ("1", "true", "yes"),
    security_token=os.environ.get("SALESFORCE_SECURITY_TOKEN") or None,
    use_sandbox=os.environ.get("SALESFORCE_USE_SANDBOX", "").lower() in ("1", "true", "yes"),
    token_url=os.environ.get("SALESFORCE_TOKEN_URL") or "https://proxi.my.salesforce.com",
)
INSTANCE_URL = token["instance_url"]
ACCESS_TOKEN = token["access_token"]
print("Authenticated:", INSTANCE_URL)

Authenticated: https://proxi.my.salesforce.com


In [12]:
from utils.sf_job_payload import prepare_payload_for_write
from utils.sf_job_rest_minimal import describe_sobject

JOB_OBJECT = os.environ.get("SALESFORCE_JOB_OBJECT", "Job__c").strip()
describe = describe_sobject(INSTANCE_URL, ACCESS_TOKEN, JOB_OBJECT)

fields = prepare_payload_for_write(
    job_row,
    describe,
    use_canonical_description=True,
    for_update=False,
    description_use_html=True,
)

# Mark as test record so it's clearly distinguishable in Salesforce
TEST_PREFIX = "[TEST] "
if "Name" in fields:
    fields["Name"] = TEST_PREFIX + (fields["Name"] or "")
if "External_Job_ID__c" in fields:
    fields["External_Job_ID__c"] = "TEST-" + (fields.get("External_Job_ID__c") or "")
if "Job_Client_Job_Description__c" in fields:
    fields["Job_Client_Job_Description__c"] = "[TEST RECORD] " + (fields["Job_Client_Job_Description__c"] or "")

print(f"{len(fields)} fields:", sorted(fields.keys()))

Skipped (not createable on object): Job_Insight__c, Job_Facility_Display__c, Job_Street_Address__c, Job_Point_of_Contact__c, Job_Standard_Schedule__c, Job_Provider_Start_Date__c, Job_Provider_End_Date__c, Position_Type_DJC__c, Specialty_DJC__c, Occupation_DJC__c, Worksite_Parent__c
16 fields: ['External_Job_ID__c', 'External_Job_Link__c', 'Job_Account__c', 'Job_City__c', 'Job_Client_Job_Description__c', 'Job_Dates_Needed__c', 'Job_Patient_Ages__c', 'Job_Ranking__c', 'Job_Recruitment_Level__c', 'Job_State__c', 'Job_Status__c', 'Job_Support_Staff__c', 'Job_Types_of_Cases__c', 'Job_Volume__c', 'Job_Worksite_Location_1__c', 'Salary_Pay_Range__c']


Note: Job_Recruitment_Level__c value 'Normal' not allowed; using first active picklist value 'Building Roster'


In [13]:
from utils.sf_job_rest_minimal import create_job_record

show = dict(fields)
dk = "Job_Client_Job_Description__c"
if dk in show and len(str(show[dk])) > 500:
    show[dk] = str(show[dk])[:500] + f"... ({len(str(fields[dk]))} chars)"
print(json.dumps(show, indent=2, default=str))

if DRY_RUN:
    print("\nDRY_RUN — set DRY_RUN = False in cell 2 and re-run from there.")
else:
    result = create_job_record(INSTANCE_URL, ACCESS_TOKEN, JOB_OBJECT, fields)
    new_id = result.get("id", "(unknown)")
    print(f"\nCreated {JOB_OBJECT} → {new_id}")

{
  "External_Job_ID__c": "TEST-19448",
  "Job_Account__c": "0015f00000HH63kAAD",
  "Job_Worksite_Location_1__c": "001UP00000HKOXRYA5",
  "Job_Client_Job_Description__c": "<p>Proxi Dental Staffing is seeking a General Dentist for a locum tenens opportunity in Syracuse, New York.</p><p>This position offers the opportunity to practice comprehensive general dentistry with a supportive clinical team and steady patient flow.</p><p><em>Travel and lodging may be available for qualified candidates.</em></p><p><strong>Dates</strong><br/>April 23-24</p><p><strong>Schedule</strong><br/>7a-5p lunch 12p-1p unpaid, Fri 7a-12p</p><p><strong>Pay range</strong><br/>Starting at $12... (2171 chars)",
  "External_Job_Link__c": "https://portal.kimedics.com/app/workspace/job-posts/19448",
  "Job_Status__c": "Closed",
  "Job_State__c": "New York",
  "Job_City__c": "Syracuse",
  "Job_Recruitment_Level__c": "Building Roster",
  "Job_Dates_Needed__c": "April 23-24",
  "Job_Types_of_Cases__c": "Surgical extracti